# Dataloader

In [2]:
import argparse
import inspect
import logging
import math
import os
import shutil
from datetime import timedelta
from pathlib import Path

import datasets
from datasets import load_dataset
import torch
import torch.nn.functional as F
from torchvision import transforms


In [3]:
train_data_dir = "/media/aris/Data/master2025dev/datasets/wood/4_main_categories/"

dataset = load_dataset("imagefolder", data_dir=train_data_dir, split="train")

In [4]:
len(dataset)
print(dataset)
png_image = dataset[0]['image']
# Extract image shape
print(png_image.size)  # Output: (width, height)

Dataset({
    features: ['image'],
    num_rows: 5954
})
(1920, 1200)


In [5]:
num_workers = os.cpu_count() // 2 # Use half of available CPU cores
batch_size = 16
# --- Basic image augmentations --- #
# VAE Resolution
resolution = 128
center_crop = True
random_flip = True

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
      
    

class dataloaderInterface:
    def __init__(self, config: str, output_dir: str, resolution: int = None, center_crop: bool = None, random_flip: bool = True, preview: bool = False, batch_size: int = None, num_workers: int = None):
        
        # Initialize config as json object
        with open(config, 'r') as f:            
            config = json.load(f)
            self.data_config = config["data"]
        
        # Set parameters from config file if not provided as arguments
        if output_dir is None:
            output_dir = config["logging"]["output_dir"]
        if batch_size is None:
            batch_size = config["hyperparameters"]["batch_size"]
        if num_workers is None:
            num_workers = config["hyperparameters"]["dataloader_num_workers"]
            
        self.output_dir = output_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        
        self.preview = preview
        
        # --------------------------------- Load data -------------------------------- #
        train_dict = self.generate_data_dict(split="train")
        val_dict = self.generate_data_dict(split="val")
        
        data_dict = train_dict + val_dict
        data_json_path = os.path.join("/tmp/", "data_dict.json")
        
        # Dump data files to json file
        with open(data_json_path, 'w') as f:
            json.dump(data_dict, f, indent=4)
        
        # Loade dataset from json file
        dataset = load_dataset("json", data_files=data_json_path)
        self.train_ds = dataset["train"].filter(lambda x: x["split"] == "train")
        self.val_ds = dataset["train"].filter(lambda x: x["split"] == "val")
        

        
        # ------------------------------- Augmentations ------------------------------ #
        # Check if augmentations args are provided, if not, use the ones from the config file
        if resolution is None:
            resolution = self.data_config["resolution"]
        if center_crop is None:
            center_crop = self.data_config["center_crop"]
        if random_flip is None:
            random_flip = self.data_config["random_flip"]
        # Preprocessing the datasets and DataLoaders creation.
        spatial_augmentations = [
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution) if center_crop else transforms.RandomCrop(resolution),
            transforms.RandomHorizontalFlip() if random_flip else transforms.Lambda(lambda x: x),
        ]

        self.augmentations = transforms.Compose(
            spatial_augmentations
            + [
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ]
        )
        
# ----------------- Generate data dictionary from config file ---------------- #
    def generate_data_dict(self, split: str = "train"):
        """Generate a data dictionary from the config file. The data dictionary will contain the filepaths, labels and split for each image in the dataset.
        arguments:
            split: str: the split to generate the data dictionary for (train, val, test)
        returns:        
            data_dict: list: a list of dictionaries containing the filepaths, labels and split for each image in the dataset
        """
        
        data_dict = []
        for category, details in self.data_config["classes"].items():
            source = details["data_dir"] + "/" + split
            sub_categories = details["sub_categories"]
            
            for sub_category in sub_categories:
                # Find all .png files in the source directory for the sub-category
                sub_category_path = os.path.join(source, sub_category)
                # Look globally in the folder in recursive way for .png files
                for root, dirs, files in os.walk(sub_category_path):
                    for file in files:
                        if file.endswith(".png"):
                            
                            image_file = os.path.join(root, file)
                            sample = {"filepath": image_file, "label": sub_category, "split": split}
                            data_dict.append(sample)  
        return data_dict
                              
# ------------- Transform images using the defined augmentations ------------- #
    def transform_images(self, examples):
        processed = []
        for image in examples["image"]:
            processed.append(self.augmentations(image.convert("RGB")))

        return {"input": processed}
    
# ------------------------------ Get dataloader ------------------------------ #
    def get_dataloader(self, split="train"):
        if split == "train":
            dataset = self.train_ds
        elif split == "val":
            dataset = self.val_ds
        else:
            raise ValueError("Invalid split. Must be 'train' or 'val'.")
            
        dataset.set_transform(self.transform_images)
        self.dataloader = torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
        if self.preview == True:
            self.preview_dataloader(self.dataloader)
        return self.dataloader
        
        
# -------------------------------- Get classes ------------------------------- #
    def get_classes(self, type="wood"):
        """ Get categories from config file 
        arguments:
            type: str: type of data to get categories for. ("wood", "plastic" etc.)
        returns:
            categories: list: list of categories
        """
        
        categories = list(self.data_config[type]["sub_categories"])
        return categories
    
    
# ---------------------------- Preview dataloader ---------------------------- #
    def preview_dataloader(self, dataloader, num_images=8):
        """
        Preview images from the dataloader in a grid.
        
        Args:
            dataloader: PyTorch DataLoader object
            num_images: Number of images to display (default: 8)
        """
        # Get a batch from the dataloader
        batch = next(iter(dataloader))
        images = batch["input"]
        
        # Limit to num_images
        images = images[:num_images]
        
        # Denormalize images from [-1, 1] to [0, 1]
        images = (images + 1) / 2
        images = torch.clamp(images, 0, 1)
        
        # Create grid
        grid_size = int(np.ceil(np.sqrt(num_images)))
        fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
        axes = axes.flatten()
        
        for idx, ax in enumerate(axes):
            if idx < len(images):
                # Convert to numpy and transpose from CxHxW to HxWxC
                img = images[idx].permute(1, 2, 0).numpy()
                ax.imshow(img)
                ax.axis('off')
            else:
                ax.axis('off')
        
        plt.tight_layout()
        plt.show()

    
    
        
    
config_path = "/media/aris/Data/master2025dev/aris_master/training/template/config.json"
output_dir = None
dataloader_interface = dataloaderInterface(config = config_path, output_dir=output_dir, preview=True)
dataloader_interface.get_dataloader(split="train")

Generating train split: 10417 examples [00:00, 221965.15 examples/s]
Filter: 100%|██████████| 10417/10417 [00:00<00:00, 249780.27 examples/s]


In [7]:


# Preprocessing the datasets and DataLoaders creation.
spatial_augmentations = [
    transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(resolution) if center_crop else transforms.RandomCrop(resolution),
    transforms.RandomHorizontalFlip() if random_flip else transforms.Lambda(lambda x: x),
]

augmentations = transforms.Compose(
    spatial_augmentations
    + [
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)


def transform_images(examples):
    processed = []
    for image in examples["image"]:
        processed.append(augmentations(image.convert("RGB")))

    return {"input": processed}


dataset.set_transform(transform_images)

train_dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
)

In [7]:
print(train_dataloader)

In [8]:
import numpy as np

import matplotlib.pyplot as plt

def preview_dataloader(dataloader, num_images=8):
    """
    Preview images from the dataloader in a grid.
    
    Args:
        dataloader: PyTorch DataLoader object
        num_images: Number of images to display (default: 8)
    """
    # Get a batch from the dataloader
    batch = next(iter(dataloader))
    images = batch["input"]
    
    # Limit to num_images
    images = images[:num_images]
    
    # Denormalize images from [-1, 1] to [0, 1]
    images = (images + 1) / 2
    images = torch.clamp(images, 0, 1)
    
    # Create grid
    grid_size = int(np.ceil(np.sqrt(num_images)))
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    axes = axes.flatten()
    
    for idx, ax in enumerate(axes):
        if idx < len(images):
            # Convert to numpy and transpose from CxHxW to HxWxC
            img = images[idx].permute(1, 2, 0).numpy()
            ax.imshow(img)
            ax.axis('off')
        else:
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Preview the dataloader
preview_dataloader(train_dataloader, num_images=16)

ERROR:tornado.general:SEND Error: Host unreachable


KeyboardInterrupt: 